# `collab-sol` utilities demo

This minimal Colab example loads the final crossing-year dataset, creates leakage-safe temporal folds, tunes an ordinary scikit-learn model, and evaluates continuous risk scores on the untouched 2024 holdout. To try another model, replace only the two functions in **Define the model**.

In [ ]:
from pathlib import Path

# Run from the repo root or notebooks/; paths resolve relative to the repo.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATASET_PATH = PROJECT_ROOT / "final_dataset_sample_50k.parquet"  # swap in final_dataset.parquet for the full file
RUN_DIR = PROJECT_ROOT / "results" / "logistic_demo"
RUN_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
import json

import pandas as pd
import pyarrow.parquet as pq
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from collab_sol import (
    generate_risk_report,
    make_expanding_temporal_split,
    tune_model,
)

## Load a small feature panel

This feature list is only a runnable demonstration, not the final feature-selection decision. Outcome columns such as `incident_count`, injuries, fatalities, and severity must never be risk features.

In [ ]:
ALL_COLUMNS = pq.read_schema(DATASET_PATH).names

DEMO_CROSSING_FEATURES = [
    "trainsperweek",
    "maximumtimetablespeed",
    "numberofmaintracks",
    "trafficlane",
    "highwayspeedlimit",
    "annualaveragedailytrafficcount",
    "estimatedpercenttrucks",
]

WEATHER_FEATURES = [
    "pr_total_mm_5y_mean",
    "wet_days_5y_mean",
    "freeze_thaw_days_5y_mean",
    "freezing_days_5y_mean",
    "hot_days_5y_mean",
    "tmin_mean_c_5y_mean",
    "tmax_mean_c_5y_mean",
    "wind_mean_ms_5y_mean",
    "high_wind_days_5y_mean",
]

BROAD_CROSSING_NUMERIC_FEATURES = [
    "latitude",
    "longitude",
    "numberpassengertrainperday",
    "totaldaylightthrutrains",
    "totalnighttimethrutrains",
    "totalswitchingtrains",
    "totaltransittrains",
    "trainsperweek",
    "maximumtimetablespeed",
    "minimumspeedrangeovercrossing",
    "maximumspeedrangeovercrossing",
    "numberofmaintracks",
    "numberofsidingtracks",
    "numberofyardtracks",
    "numberoftransittracks",
    "numberofindustrytracks",
    "numbercrossbuckassemblies",
    "numberstopsigns",
    "numberyieldsigns",
    "numberlowgroundclearancesigns",
    "countroadwaygatearms",
    "countpedestriangatearms",
    "countcantileveroverlane",
    "countcantilevernolane",
    "countmastmountedflashinglight",
    "countflashinglightpair",
    "numberofbells",
    "countotherflashinglights",
    "trafficlane",
    "intersectingroadwaydistance",
    "highwayspeedlimit",
    "annualaveragedailytrafficcount",
    "estimatedpercenttrucks",
    "schoolbusaveragecount",
]

ACS_PREFIXES = ("B01003_", "B01001_", "B03002_", "B19013_", "B19001_", "B15003_", "B08301_", "B08303_")
ACS_FEATURES = [column for column in ALL_COLUMNS if column.startswith(ACS_PREFIXES)]

# Categorical fields need one-hot encoding for logistic regression. Neural models can use embeddings instead.
CATEGORICAL_CANDIDATE_FEATURES = [
    "reportingagencytypecode",
    "statecode",
    "innearcode",
    "crossingpurposecode",
    "typeoftrainserviceid1",
    "developmenttypecode",
    "whistlebancode",
    "latlongsourcecode",
    "tracksignaled",
    "eventrecorder",
    "remotehealthmonitoring",
    "signsorsignals",
    "lowgroundclearancesigns",
    "exemptsigns",
    "enssigndisplayed",
    "othermutcdsigns",
    "waysidehorn",
    "highwaytrafficsignal",
    "nearbyhighwaytrafficsignals",
    "highwaytrafficpresignals",
    "trafficlanetypecode",
    "highwaypaved",
    "trackrundownstreet",
    "crossingilluminated",
    "crossingsurfaceid1",
    "highwaynear",
    "smallestcrossinganglecode",
    "compower",
    "highwaysystemcode",
    "roadatcrossingcode",
    "roadatcrossingtypecode",
    "crossingonstatehighwaysystem",
    "highwayspeedlimittypecode",
    "schoolbusroute",
    "emergencyservicesroute",
    "reporting_railroad_class",
    "reporting_railroad_smt_grouping",
    "wdcode",
]

DEMO_CATEGORICAL_FEATURES = [
    "statecode",
    "crossingpurposecode",
    "developmenttypecode",
    "tracksignaled",
    "signsorsignals",
    "highwaypaved",
    "roadatcrossingcode",
    "crossingonstatehighwaysystem",
]

# Outcomes would leak information and must never be included as risk features.
OUTCOME_COLUMNS = [
    "incident",
    "incident_count",
    "total_killed",
    "total_injured",
    "severity_ordinal",
    "severity_target",
]

# Dataset keys and processing metadata are retained for auditing, not modeling.
DATASET_METADATA_COLUMNS = [
    "crossingid",
    "asof_year",
    "revisiondate",
    "geoid",
    "GEOID",
    "acs_year",
    "gridmet_mapped",
    "gridmet_distance_km",
    "grid_lat_index",
    "grid_lon_index",
    "grid_latitude",
    "grid_longitude",
    "weather_year_start",
    "weather_year_end",
    "gridmet_data_available",
]

# These are constant after filtering to open, public, at-grade crossings.
CONSTANT_COLUMNS = [
    "crossingtypecode",
    "crossingtype",
    "crossingpositioncode",
    "crossingposition",
    "advancewarningtext7",
    "pavementmarking4",
    "reporttype",
    "crossingclosed",
    "reportstatus",
    "closed_flag",
]

EXCLUDE_FROM_RISK_FEATURES = sorted(set(
    OUTCOME_COLUMNS
    + DATASET_METADATA_COLUMNS
    + CONSTANT_COLUMNS
))

FEATURE_SETS = {
    "demo": {
        "numeric": DEMO_CROSSING_FEATURES + WEATHER_FEATURES,
        "categorical": DEMO_CATEGORICAL_FEATURES,
    },
    "broad": {
        "numeric": list(dict.fromkeys(BROAD_CROSSING_NUMERIC_FEATURES + WEATHER_FEATURES + ACS_FEATURES)),
        "categorical": CATEGORICAL_CANDIDATE_FEATURES,
    },
}
FEATURE_SET = "demo"  # Change to "broad" after the smoke test works.
NUMERIC_FEATURES = FEATURE_SETS[FEATURE_SET]["numeric"]
CATEGORICAL_FEATURES = FEATURE_SETS[FEATURE_SET]["categorical"]
FEATURE_COLUMNS = NUMERIC_FEATURES + CATEGORICAL_FEATURES

missing_columns = sorted(set(FEATURE_COLUMNS) - set(ALL_COLUMNS))
if missing_columns:
    raise ValueError(f"Missing selected columns: {missing_columns}")

print(f"Using {FEATURE_SET!r}: {len(NUMERIC_FEATURES)} numeric + {len(CATEGORICAL_FEATURES)} categorical features")
print(f"Hard exclusions: {len(EXCLUDE_FROM_RISK_FEATURES)} columns")

frame = pd.read_parquet(
    DATASET_PATH,
    columns=["crossingid", "asof_year", "incident", *FEATURE_COLUMNS],
)
display(frame.groupby("asof_year")["incident"].agg(rows="size", crashes="sum", prevalence="mean"))

## Create the temporal split

With 2019–2024 data, Optuna validates on 2020, 2021, 2022, and 2023. The utility reserves 2024 as the untouched final test year. Inside every fold, numeric columns are imputed and scaled while categorical columns are imputed and one-hot encoded. A fresh preprocessor is fit using only that fold's training years.

In [ ]:
X = frame[FEATURE_COLUMNS]
y = frame["incident"].astype(int)
split = make_expanding_temporal_split(frame, year_column="asof_year")

def make_preprocessor():
    return ColumnTransformer([
        (
            "numeric",
            make_pipeline(
                SimpleImputer(strategy="median", add_indicator=True),
                StandardScaler(),
            ),
            NUMERIC_FEATURES,
        ),
        (
            "categorical",
            make_pipeline(
                SimpleImputer(strategy="most_frequent"),
                OneHotEncoder(handle_unknown="ignore"),
            ),
            CATEGORICAL_FEATURES,
        ),
    ])

print("Validation years:", [fold.validation_year for fold in split.folds])
print("Untouched test year:", split.holdout_year)

In [ ]:
split.folds[0].train_indices

## Define the model

These are the only two model-specific functions required by the tuner. `build_model` can return a teammate's own class as long as it implements `fit(X, y)` and `predict_proba(X)`.

In [ ]:
# Put all of the parameters you want Optuna to tune here.
def suggest_params(trial):
    return {"C": trial.suggest_float("C", 1e-3, 100, log=True)}

def build_model(params, random_state):
    return LogisticRegression(
        **params,
        class_weight="balanced",
        max_iter=300,
        random_state=random_state,
    )

In [ ]:
N_TRIALS = 2  # Smoke test only; increase for the real comparison.

study = tune_model(
    model_factory=build_model,
    suggest_params=suggest_params,
    X=X,
    y=y,
    split=split,
    preprocessor_factory=make_preprocessor,
    n_trials=N_TRIALS,
    study_name="logistic_risk",
)

study.trials_dataframe().to_csv(RUN_DIR / "optuna_trials.csv", index=False)
print("Mean validation AP:", study.best_value)
print("Best parameters:", study.best_params)

## Refit once and evaluate 2024

Optuna is finished before this cell touches the holdout. The output `risk_scores.csv` contains the continuous score used to prioritize crossings, not a yes/no prediction.

In [ ]:
development_rows = split.development_indices
test_rows = split.holdout_indices

preprocessor = make_preprocessor()
X_development = preprocessor.fit_transform(X.iloc[development_rows], y.iloc[development_rows])
X_test = preprocessor.transform(X.iloc[test_rows])

model = build_model(study.best_params, random_state=42)
model.fit(X_development, y.iloc[development_rows].to_numpy())

metrics = generate_risk_report(
    model,
    X_test,
    y.iloc[test_rows].to_numpy(),
    output_dir=RUN_DIR,
    row_ids=frame.iloc[test_rows]["crossingid"].to_numpy(),
)

run_metadata = {
    "model": "logistic_regression",
    "holdout_year": split.holdout_year,
    "features": FEATURE_COLUMNS,
    "numeric_features": NUMERIC_FEATURES,
    "categorical_features": CATEGORICAL_FEATURES,
    "best_params": study.best_params,
}
(RUN_DIR / "run.json").write_text(json.dumps(run_metadata, indent=2))

display(pd.DataFrame([metrics.to_dict()]))
print("Report saved to:", RUN_DIR)